In [ ]:
import pandas as pd

sales = pd.read_csv(
    "/kaggle/input/competitions/m5-forecasting-accuracy/sales_train_validation.csv"
)

print(sales.shape)
sales.head()

In [ ]:
row = sales.iloc[0]

print(row[:10])



In [ ]:
series = row.iloc[6:]

series = series.astype(float)

print(series.head())
print(series.shape)

In [ ]:
import os
import matplotlib.pyplot as plt

os.makedirs("visualizations", exist_ok=True)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))

plt.plot(series.values)

plt.title("Daily Sales")

plt.xlabel("Days")

plt.ylabel("Units Sold")

plt.show()

In [ ]:
non_zero_days = (series > 0).sum()

total_days = len(series)

print("Non-zero sales days:", non_zero_days)

print("Total days:", total_days)

print("Percentage active:", 
      round((non_zero_days / total_days) * 100, 2), "%")

In [ ]:
sales_only = sales.iloc[:, 6:]

active_counts = (sales_only > 0).sum(axis=1)

sales["active_percentage"] = (
    active_counts / sales_only.shape[1]
) * 100


In [ ]:
top_products = sales.sort_values(
    by="active_percentage",
    ascending=False
)

top_products[
    [
        "id",
        "active_percentage"
    ]
].head(10)

In [ ]:
target_row = sales[
    sales["id"] == "FOODS_3_586_CA_2_validation"
]

target_row = target_row.iloc[0]

In [ ]:
series = target_row.iloc[6:-1]

series = series.astype(float)

print(series.head())

print(series.shape)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))

plt.plot(series.values)

plt.title("FOODS_3_586_CA_2 Sales")

plt.xlabel("Days")

plt.ylabel("Units Sold")

plt.show()

In [ ]:
train = series[:-28]

test = series[-28:]

print("Train size:", len(train))

print("Test size:", len(test))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))

plt.plot(train.values, label="Train")

plt.plot(
    range(len(train), len(series)),
    test.values,
    label="Test"
)

plt.legend()

plt.title("Train/Test Split")

plt.show()

In [ ]:
series.index = pd.RangeIndex(
    start=0,
    stop=len(series)
)

train = series[:-28]

test = series[-28:]

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(
    train,
    order=(1,1,1),
    seasonal_order=(1,1,1,7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

result = model.fit()

In [ ]:
print(result.summary())

In [ ]:
forecast = result.forecast(steps=28)

print(forecast.head())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))

plt.plot(
    test.index,
    test.values,
    label="Actual"
)

plt.plot(
    test.index,
    forecast.values,
    label="Forecast"
)

plt.legend()

plt.title("SARIMA Forecast vs Actual")

plt.show()

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

mae = mean_absolute_error(
    test,
    forecast
)

rmse = mean_squared_error(
    test,
    forecast
) ** 0.5

print("MAE:", mae)

print("RMSE:", rmse)

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(series.values)

plt.title("Complete Sales History")

plt.xlabel("Days")

plt.ylabel("Units Sold")

plt.show()

In [ ]:
rolling_mean = series.rolling(window=30).mean()

plt.figure(figsize=(15,5))

plt.plot(series.values, alpha=0.4, label="Daily Sales")

plt.plot(
    rolling_mean.values,
    label="30-Day Rolling Mean"
)

plt.legend()

plt.title("Sales Trend with Rolling Average")

plt.show()

In [ ]:
weekly_pattern = []

for i in range(7):
    weekly_pattern.append(
        series[i::7].mean()
    )

days = [
    "Day 1",
    "Day 2",
    "Day 3",
    "Day 4",
    "Day 5",
    "Day 6",
    "Day 7"
]

plt.figure(figsize=(10,5))

plt.bar(days, weekly_pattern)

plt.title("Average Weekly Sales Pattern")

plt.ylabel("Average Units Sold")

plt.show()

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(
    train.index[-100:],
    train.values[-100:],
    label="Train"
)

plt.plot(
    test.index,
    test.values,
    label="Actual"
)

plt.plot(
    forecast.index,
    forecast.values,
    label="Forecast"
)

plt.legend()

plt.title("SARIMA Forecasting")

plt.show()